# 06 — FX exploration, regime-conditional correlations, holdout robustness

Three builds that close the remaining research gaps before the dashboard:

- **Part A — FX track.** The FX panels were built but never analyzed. Safe-haven
  co-movement (CHF/JPY/gold in stress), and dollar-vs-commodities as two sides
  of one coin.
- **Part B — Regime-conditional correlations.** Generalize the gold-real-yield
  regime finding: does the *whole* correlation structure reshape in high-vol?
  (The classic "diversification fails in crises" question.)
- **Part C — Holdout robustness.** Dollar-dominance and the yield-split were
  explore-only. Do they hold on 2022+? The honesty capstone.

**Quote-convention note:** FX columns mix conventions — `usd_eur`/`usd_gbp` are
USD-per-foreign (up = foreign currency stronger), while `jpy_usd`/`cny_usd`/
`chf_usd` are foreign-per-USD (up = USD stronger). Signs are interpreted with
this in mind.

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, str(Path.cwd().parent))
import config
plt.rcParams["figure.figsize"]=(11,5)

comm = pd.read_parquet(config.DATA/"commodities_monthly.parquet")
fx   = pd.read_parquet(config.DATA/"fx_monthly.parquet")
FX=["usd_eur","jpy_usd","usd_gbp","cny_usd","chf_usd","usd_broad"]
CORE=["gold","silver","wti","brent","natgas","copper","palladium","platinum"]
print("comm:", comm.shape, "| fx:", fx.shape)

## Part A1 — FX return correlation structure

Monthly FX returns among the majors. Watch for the safe-haven block: in this
data, does the yen/franc move together, and against the risk currencies?

In [ ]:
fx_ret = fx[FX].pct_change()
corr = fx_ret.corr()
fig,ax=plt.subplots(figsize=(6.5,5.5))
im=ax.imshow(corr,cmap="RdBu_r",vmin=-1,vmax=1)
ax.set_xticks(range(len(FX))); ax.set_xticklabels(FX,rotation=45,ha="right")
ax.set_yticks(range(len(FX))); ax.set_yticklabels(FX)
for i in range(len(FX)):
    for j in range(len(FX)):
        ax.text(j,i,f"{corr.iloc[i,j]:.2f}",ha="center",va="center",fontsize=8)
plt.colorbar(im,fraction=0.046); plt.title("FX monthly-return correlations")
plt.tight_layout(); plt.show()

## Part A2 — The safe-haven question

Do gold, the yen, and the franc co-move as havens? Build a small cross-corr of
gold returns vs each FX return, then check whether the haven currencies
strengthen when equity-vol (VIX) spikes.

In [ ]:
gold_ret = np.log(comm["gold"].where(comm["gold"]>0)).diff()
vix_chg  = comm["vix"].diff()
haven = pd.DataFrame({"gold":gold_ret})
for f in FX:
    haven[f] = fx_ret[f]
haven["vix_chg"] = vix_chg
hc = haven.corr()
print("Correlation of gold and FX returns with VIX change (risk-off signal):")
print(hc["vix_chg"].drop("vix_chg").round(3).to_string())
print("\nNote sign vs quote convention: for jpy_usd/chf_usd (foreign-per-USD),")
print("a NEGATIVE corr with VIX means USD-down / franc-yen-UP in risk-off = haven bid.")

## Part A3 — Dollar and commodities: two sides of one coin

The research found every commodity is strongly negative to the broad dollar.
Confirm from the FX side: the broad-USD return vs the commodity complex return.

In [ ]:
usd_ret = fx["usd_broad"].pct_change()
comm_ret = np.log(comm[CORE].where(comm[CORE]>0)).diff()
usd_vs_comm = pd.Series({c: usd_ret.corr(comm_ret[c]) for c in CORE}).sort_values()
print("Broad-USD return vs commodity returns (expect all negative):")
print(usd_vs_comm.round(3).to_string())

## Part B — Regime-conditional correlations

Split the commodity+macro correlation matrix by volatility regime. If
correlations rise toward 1 in high-vol, that's the "diversification fails when
you need it" effect — visually and practically important.

In [ ]:
assets = CORE + ["usd_broad","ust10y_real","vix"]
allret = pd.DataFrame(index=comm.index)
for a in CORE: allret[a]=np.log(comm[a].where(comm[a]>0)).diff()
allret["usd_broad"]=comm["usd_broad"].diff()
allret["ust10y_real"]=comm["ust10y_real"].diff()
allret["vix"]=comm["vix"].diff()

mkt_vol = allret[CORE].rolling(12).std().mean(axis=1)
cutoff = mkt_vol.loc[:config.EXPLORE_END].quantile(0.70)
regime = (mkt_vol>cutoff)

corr_calm = allret[~regime].corr()
corr_high = allret[regime].corr()

fig,axes=plt.subplots(1,2,figsize=(15,6))
for ax,(cm,title) in zip(axes,[(corr_calm,"CALM"),(corr_high,"HIGH-VOL")]):
    im=ax.imshow(cm,cmap="RdBu_r",vmin=-1,vmax=1)
    ax.set_xticks(range(len(assets))); ax.set_xticklabels(assets,rotation=45,ha="right",fontsize=8)
    ax.set_yticks(range(len(assets))); ax.set_yticklabels(assets,fontsize=8)
    ax.set_title(f"{title} regime correlations")
plt.colorbar(im,ax=axes,fraction=0.023); plt.show()

In [ ]:
# quantify: average absolute correlation among the 8 commodities, by regime
def avg_abs_corr(cm, cols):
    sub=cm.loc[cols,cols].values
    iu=np.triu_indices(len(cols),1)
    return np.abs(sub[iu]).mean()
print(f"avg |corr| among commodities, CALM:     {avg_abs_corr(corr_calm,CORE):.3f}")
print(f"avg |corr| among commodities, HIGH-VOL: {avg_abs_corr(corr_high,CORE):.3f}")
print("\nIf high-vol is meaningfully higher, commodities move together more in")
print("stress -- diversification within the complex weakens exactly in crises.")

## Part C — Holdout robustness of the headline findings

The dollar-dominance and yield-split were established on explore only. Re-run
them on the untouched holdout (2022+). If signs and rough magnitudes hold, the
factor structure is stable, not a pre-2022 artifact.

In [ ]:
def macro_corr(returns_df, price_ret, macro_chg):
    return price_ret.corr(macro_chg)

ex = comm.loc[:config.EXPLORE_END]; ho = comm.loc[config.HOLDOUT_START:]
def block(df):
    pr = {c: np.log(df[c].where(df[c]>0)).diff() for c in CORE}
    usd = df["usd_broad"].diff(); ynom = df["ust10y_nominal"].diff()
    out={}
    for c in CORE:
        out[c] = {"vs_usd": pr[c].corr(usd), "vs_nominal_yield": pr[c].corr(ynom)}
    return pd.DataFrame(out).T

ex_b, ho_b = block(ex), block(ho)
compare = pd.DataFrame({
    "usd_explore": ex_b["vs_usd"].round(2), "usd_holdout": ho_b["vs_usd"].round(2),
    "yield_explore": ex_b["vs_nominal_yield"].round(2), "yield_holdout": ho_b["vs_nominal_yield"].round(2),
})
compare

In [ ]:
# verdict checks
usd_stable = ((ex_b["vs_usd"]<0) & (ho_b["vs_usd"]<0)).sum()
print(f"Dollar-dominance: negative in BOTH explore & holdout for {usd_stable}/8 assets")
# yield split: precious (gold,silver) negative; energy/industrial (wti,brent,copper) positive
prec = ["gold","silver"]; grow = ["wti","brent","copper"]
prec_ok = all(ho_b.loc[p,"vs_nominal_yield"]<0.15 for p in prec)
grow_ok = all(ho_b.loc[g,"vs_nominal_yield"]>-0.05 for g in grow)
print(f"Yield-split holds in holdout: precious still <=~0 ({prec_ok}), growth still >=~0 ({grow_ok})")

## Summary (fill after running)

- **FX/safe-haven:** do franc/yen show haven behavior vs VIX? gold too?
- **USD two-sides:** all commodities negative to broad USD from the FX side?
- **Regime correlations:** avg |corr| calm vs high-vol — does diversification fail in stress?
- **Holdout robustness:** dollar-dominance stable in N/8? yield-split intact?

**Dashboard implication:** confirmed-stable factor structure earns headline
billing; the calm-vs-high-vol correlation pair is a striking dashboard visual;
FX adds a safe-haven context panel. Anything that DIDN'T survive holdout gets
labeled explore-only.